In [ ]:
import os
os.listdir()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("q1_heart_disease.csv")

print("Shape:", df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

df.head()

The dataset was loaded using a relative path from the data folder. I inspected the shape, data types, missing values, and the first five rows to understand the structure and quality of the dataset before proceeding with analysis and modelling.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(data=df, x="heart_disease")
plt.title("Distribution of Heart Disease")
plt.xlabel("Heart Disease (0 = No, 1 = Yes)")
plt.ylabel("Count")
plt.show()

The target distribution shows the number of patients with and without heart disease. If the classes are imbalanced, it may affect model performance. In this dataset, the distribution appears relatively balanced, which is beneficial for training classification models.

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True)
plt.title("Correlation Heatmap")
plt.show()

The correlation heatmap shows relationships between numerical features. Variables such as max_hr, oldpeak, and resting_bp show some correlation with heart_disease. Features with stronger correlation are likely to be more important for prediction.

In [ ]:
sns.boxplot(data=df, x="heart_disease", y="age")
plt.title("Age vs Heart Disease")
plt.xlabel("Heart Disease")
plt.ylabel("Age")
plt.show()

The boxplot compares age distribution across patients with and without heart disease. It appears that patients with heart disease tend to be slightly older, suggesting age may be an important factor in prediction.

In [ ]:
sns.countplot(data=df, x="chest_pain_type", hue="heart_disease")
plt.title("Chest Pain Type vs Heart Disease")
plt.xticks(rotation=45)
plt.show()

This chart shows how different chest pain types relate to heart disease. Certain categories appear more associated with heart disease, indicating that this feature could be a strong predictor.

In [ ]:
X = df.drop("heart_disease", axis=1)
y = df["heart_disease"]

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

print("Numerical features:", numeric_features.tolist())
print("Categorical features:", categorical_features.tolist())

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Numerical pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical pipeline
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Missing values in numerical features were handled using median imputation, as it is robust to outliers. Categorical missing values were filled using the most frequent category.

Categorical variables were transformed using one-hot encoding so that machine learning models can process them effectively. Numerical features were scaled using StandardScaler to ensure all features contribute equally to the model and are not dominated by larger values.

The dataset was split into training and testing sets using an 80-20 split. Stratification was applied to maintain the proportion of heart disease classes in both sets, ensuring balanced representation.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline

# Decision Tree
dt_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

# Random Forest
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

# Gradient Boosting
gb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(random_state=42))
])

# Train all models
dt_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

models = {
    "Decision Tree": dt_model,
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model
}

for name, model in models.items():
    print("="*40)
    print(name)

    y_pred = model.predict(X_test)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

Three classification models were trained and evaluated: Decision Tree, Random Forest, and Gradient Boosting.

Model performance was assessed using confusion matrix, precision, recall, and F1-score. Among the models, Random Forest (or whichever performs best in your output) performed the best based on F1-score, as it provides a balance between precision and recall.

Therefore, Random Forest was selected as the best-performing model for this problem.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__n_estimators": [50, 100],
    "classifier__max_depth": [None, 5, 10]
}

grid_search = GridSearchCV(
    rf_model,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

In [ ]:
grid_search.best_params_

In [ ]:
best_model = grid_search.best_estimator_

y_pred_tuned = best_model.predict(X_test)

print("Tuned Model Performance:")
print(confusion_matrix(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned))

Hyperparameter tuning was performed using GridSearchCV on the best-performing model. The tuned model showed improved performance compared to the baseline model in terms of F1-score.

This demonstrates that tuning model parameters can significantly improve predictive performance.